In [2]:
import pandas as pd

# Load flight data
df = pd.read_csv( r"C:\Users\salsi\Desktop\uni\420\FinalProject\us-airline-operational-performance/data/01_OnTime_Performance/merged/OnTime_2023_2025_ALL.csv")

print(df.columns)

C:\Users\salsi\AppData\Local\Temp\ipykernel_16696\4235313363.py:4: DtypeWarning: Columns (0: ORIGIN_AIRPORT_ID, 1: DEST, 2: CRS_DEP_TIME, 3: CANCELLATION_CODE) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv( r"C:\Users\salsi\Desktop\uni\420\FinalProject\us-airline-operational-performance/data/01_OnTime_Performance/merged/OnTime_2023_2025_ALL.csv")


Index(['YEAR', 'QUARTER', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'FL_DATE',
       'OP_UNIQUE_CARRIER', 'OP_CARRIER_AIRLINE_ID', 'TAIL_NUM',
       'OP_CARRIER_FL_NUM', 'ORIGIN_AIRPORT_ID', 'ORIGIN', 'ORIGIN_CITY_NAME',
       'ORIGIN_STATE_ABR', 'ORIGIN_STATE_NM', 'DEST_AIRPORT_ID', 'DEST',
       'DEST_CITY_NAME', 'DEST_STATE_ABR', 'DEST_STATE_NM', 'CRS_DEP_TIME',
       'DEP_TIME', 'DEP_DELAY', 'DEP_DELAY_NEW', 'DEP_DEL15', 'CRS_ARR_TIME',
       'ARR_TIME', 'ARR_DELAY', 'ARR_DELAY_NEW', 'ARR_DEL15', 'CANCELLED',
       'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME',
       'ACTUAL_ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'CARRIER_DELAY',
       'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY'],
      dtype='str')


In [3]:
bad_years = df[~df['YEAR'].astype(str).str.isnumeric()]
print(bad_years.head())

Empty DataFrame
Columns: [YEAR, QUARTER, MONTH, DAY_OF_MONTH, DAY_OF_WEEK, FL_DATE, OP_UNIQUE_CARRIER, OP_CARRIER_AIRLINE_ID, TAIL_NUM, OP_CARRIER_FL_NUM, ORIGIN_AIRPORT_ID, ORIGIN, ORIGIN_CITY_NAME, ORIGIN_STATE_ABR, ORIGIN_STATE_NM, DEST_AIRPORT_ID, DEST, DEST_CITY_NAME, DEST_STATE_ABR, DEST_STATE_NM, CRS_DEP_TIME, DEP_TIME, DEP_DELAY, DEP_DELAY_NEW, DEP_DEL15, CRS_ARR_TIME, ARR_TIME, ARR_DELAY, ARR_DELAY_NEW, ARR_DEL15, CANCELLED, CANCELLATION_CODE, DIVERTED, CRS_ELAPSED_TIME, ACTUAL_ELAPSED_TIME, AIR_TIME, DISTANCE, CARRIER_DELAY, WEATHER_DELAY, NAS_DELAY, SECURITY_DELAY, LATE_AIRCRAFT_DELAY]
Index: []

[0 rows x 42 columns]


In [5]:
# Count departures by airport and month
departures = df.groupby([
    "YEAR", "QUARTER", "MONTH",
    "ORIGIN", "ORIGIN_CITY_NAME", "ORIGIN_STATE_NM"
]).size().reset_index(name="Departures_Performed")

# Count arrivals
arrivals = df.groupby([
    "YEAR", "QUARTER", "MONTH",
    "DEST", "DEST_CITY_NAME", "DEST_STATE_NM"
]).size().reset_index(name="Arrivals_Performed")

# Rename DEST fields to match ORIGIN fields for the merge
arrivals = arrivals.rename(columns={
    "DEST": "ORIGIN",
    "DEST_CITY_NAME": "ORIGIN_CITY_NAME",
    "DEST_STATE_NM": "ORIGIN_STATE_NM"
})

# Merge departures + arrivals
traffic = departures.merge(
    arrivals,
    on=["YEAR", "QUARTER", "MONTH", "ORIGIN", "ORIGIN_CITY_NAME", "ORIGIN_STATE_NM"],
    how="outer"
)

# Fill missing counts with 0 (only the count columns)
traffic[["Departures_Performed", "Arrivals_Performed"]] = traffic[
    ["Departures_Performed", "Arrivals_Performed"]
].fillna(0)

# Calculate totals
traffic["Total_Operations"] = traffic["Departures_Performed"] + traffic["Arrivals_Performed"]

# Final rename for clean output
traffic = traffic.rename(columns={
    "YEAR": "Year",
    "QUARTER": "Quarter",
    "MONTH": "Month",
    "ORIGIN": "Airport_Code",
    "ORIGIN_CITY_NAME": "City",
    "ORIGIN_STATE_NM": "State"
})

# Optional: reorder columns to match your original output order
traffic = traffic[[
    "Year", "Quarter", "Month",
    "Airport_Code", "City", "State",
    "Departures_Performed", "Arrivals_Performed", "Total_Operations"
]]


# Save
traffic.to_csv(r"C:\Users\salsi\Desktop\uni\420\FinalProject\us-airline-operational-performance/data/04_Airport_Traffic_Data/Airport_Traffic_2023_2025.csv", index=False)

print(f"✅ Created airport traffic data")
print(f"✅ Total airports: {traffic['Airport_Code'].nunique()}")

✅ Created airport traffic data
✅ Total airports: 1236
